In [9]:
# ============================================================
# 🌦️ ADVANCED WEATHER FORECAST & ALERT APPLICATION
# GLASSMORPHISM UI + MAPS + LIVE CHARTS + ALERTS
# STREAMLIT + OPEN METEO + PLOTLY + FOLIUM
# GOOGLE COLAB READY
# ============================================================

# ============================================================
# INSTALL LIBRARIES
# ============================================================

!pip install streamlit pyngrok plotly pandas requests streamlit-folium folium --quiet

# ============================================================
# STOP OLD SERVICES
# ============================================================

!pkill streamlit
!pkill ngrok

# ============================================================
# CREATE STREAMLIT APP
# ============================================================

app_code = """

import streamlit as st
import requests
import pandas as pd
import plotly.graph_objects as go
import folium

from streamlit_folium import st_folium
from datetime import datetime

# ============================================================
# PAGE CONFIG
# ============================================================

st.set_page_config(
    page_title="Weather Forecast Dashboard",
    layout="wide",
    page_icon="🌦️"
)

# ============================================================
# CUSTOM CSS
# ============================================================

st.markdown(\"\"\"
<style>

html, body, [class*="css"]  {
    background-color: #0B1120;
    color: white;
    font-family: 'Segoe UI';
}

.main {
    background: linear-gradient(135deg, #0B1120, #111827);
}

section[data-testid="stSidebar"] {
    background-color: #111827;
}

h1, h2, h3 {
    color: white;
}

.metric-card {
    background: rgba(255,255,255,0.08);
    padding: 20px;
    border-radius: 20px;
    backdrop-filter: blur(12px);
    text-align: center;
    box-shadow: 0 8px 32px rgba(0,0,0,0.3);
}

.alert-box {
    background: rgba(255, 0, 0, 0.2);
    padding: 15px;
    border-radius: 15px;
    margin-bottom: 10px;
}

.success-box {
    background: rgba(0, 255, 100, 0.2);
    padding: 15px;
    border-radius: 15px;
}

</style>
\"\"\", unsafe_allow_html=True)

# ============================================================
# TITLE
# ============================================================

st.title("🌦️ Advanced Weather Forecast Dashboard")

st.markdown(
    "### Real-Time Weather Monitoring & Smart Alert System"
)

# ============================================================
# CITY DATABASE
# ============================================================

cities = {
    "Kolkata": (22.5726, 88.3639),
    "Delhi": (28.7041, 77.1025),
    "Mumbai": (19.0760, 72.8777),
    "Chennai": (13.0827, 80.2707),
    "Bangalore": (12.9716, 77.5946),
    "Hyderabad": (17.3850, 78.4867),
    "London": (51.5072, -0.1276),
    "New York": (40.7128, -74.0060),
    "Tokyo": (35.6762, 139.6503),
    "Paris": (48.8566, 2.3522),
    "Dubai": (25.2048, 55.2708)
}

# ============================================================
# SIDEBAR
# ============================================================

st.sidebar.title("⚙️ Dashboard Settings")

selected_city = st.sidebar.selectbox(
    "🌍 Select City",
    list(cities.keys())
)

temp_threshold = st.sidebar.slider(
    "🔥 High Temperature Alert",
    30,
    50,
    40
)

humidity_threshold = st.sidebar.slider(
    "💧 Humidity Alert",
    50,
    100,
    80
)

rain_threshold = st.sidebar.slider(
    "🌧️ Rain Alert",
    10,
    100,
    60
)

# ============================================================
# GET COORDINATES
# ============================================================

lat, lon = cities[selected_city]

# ============================================================
# API CALL
# ============================================================

url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&hourly=temperature_2m,relative_humidity_2m,precipitation_probability,wind_speed_10m&current_weather=true&timezone=auto"

response = requests.get(url)

data = response.json()

# ============================================================
# EXTRACT DATA
# ============================================================

current_temp = data["current_weather"]["temperature"]
current_wind = data["current_weather"]["windspeed"]

hourly = data["hourly"]

df = pd.DataFrame({
    "time": hourly["time"],
    "temperature": hourly["temperature_2m"],
    "humidity": hourly["relative_humidity_2m"],
    "rain_probability": hourly["precipitation_probability"],
    "wind_speed": hourly["wind_speed_10m"]
})

df["time"] = pd.to_datetime(df["time"])

# ============================================================
# ALERTS
# ============================================================

alerts = []

max_temp = df["temperature"].max()
max_humidity = df["humidity"].max()
max_rain = df["rain_probability"].max()

if max_temp >= temp_threshold:
    alerts.append(f"🔥 High Temperature Alert: {max_temp}°C")

if max_humidity >= humidity_threshold:
    alerts.append(f"💧 High Humidity Alert: {max_humidity}%")

if max_rain >= rain_threshold:
    alerts.append(f"🌧️ Rain Alert: {max_rain}%")

# ============================================================
# KPI CARDS
# ============================================================

col1, col2, col3, col4 = st.columns(4)

with col1:
    st.markdown(f'''
    <div class="metric-card">
        <h3>🌡️ Temperature</h3>
        <h1>{current_temp} °C</h1>
    </div>
    ''', unsafe_allow_html=True)

with col2:
    st.markdown(f'''
    <div class="metric-card">
        <h3>💨 Wind Speed</h3>
        <h1>{current_wind} km/h</h1>
    </div>
    ''', unsafe_allow_html=True)

with col3:
    st.markdown(f'''
    <div class="metric-card">
        <h3>🔥 Max Temp</h3>
        <h1>{max_temp} °C</h1>
    </div>
    ''', unsafe_allow_html=True)

with col4:
    st.markdown(f'''
    <div class="metric-card">
        <h3>💧 Humidity</h3>
        <h1>{max_humidity}%</h1>
    </div>
    ''', unsafe_allow_html=True)

# ============================================================
# ALERT SECTION
# ============================================================

st.markdown("## 🚨 Smart Weather Alerts")

if alerts:
    for alert in alerts:
        st.markdown(
            f'<div class="alert-box">{alert}</div>',
            unsafe_allow_html=True
        )
else:
    st.markdown(
        '<div class="success-box">✅ No Severe Weather Alerts</div>',
        unsafe_allow_html=True
    )

# ============================================================
# WEATHER MAP
# ============================================================

st.markdown("## 🗺️ Live Weather Map")

weather_map = folium.Map(
    location=[lat, lon],
    zoom_start=8,
    tiles="CartoDB dark_matter"
)

folium.Marker(
    [lat, lon],
    popup=selected_city,
    tooltip=selected_city,
    icon=folium.Icon(color="blue", icon="cloud")
).add_to(weather_map)

st_folium(weather_map, width=1200, height=500)

# ============================================================
# TEMPERATURE CHART
# ============================================================

st.markdown("## 🌡️ Temperature Forecast")

temp_fig = go.Figure()

temp_fig.add_trace(go.Scatter(
    x=df["time"],
    y=df["temperature"],
    mode="lines",
    fill="tozeroy",
    name="Temperature"
))

temp_fig.update_layout(
    template="plotly_dark",
    height=500,
    paper_bgcolor="#111827",
    plot_bgcolor="#111827",
    margin=dict(l=20, r=20, t=40, b=20)
)

st.plotly_chart(temp_fig, use_container_width=True)

# ============================================================
# HUMIDITY CHART
# ============================================================

st.markdown("## 💧 Humidity Forecast")

humidity_fig = go.Figure()

humidity_fig.add_trace(go.Scatter(
    x=df["time"],
    y=df["humidity"],
    mode="lines",
    fill="tozeroy",
    name="Humidity"
))

humidity_fig.update_layout(
    template="plotly_dark",
    height=500,
    paper_bgcolor="#111827",
    plot_bgcolor="#111827",
    margin=dict(l=20, r=20, t=40, b=20)
)

st.plotly_chart(humidity_fig, use_container_width=True)

# ============================================================
# RAIN CHART
# ============================================================

st.markdown("## 🌧️ Rain Probability")

rain_fig = go.Figure()

rain_fig.add_trace(go.Bar(
    x=df["time"],
    y=df["rain_probability"],
    name="Rain"
))

rain_fig.update_layout(
    template="plotly_dark",
    height=500,
    paper_bgcolor="#111827",
    plot_bgcolor="#111827",
    margin=dict(l=20, r=20, t=40, b=20)
)

st.plotly_chart(rain_fig, use_container_width=True)

# ============================================================
# WIND SPEED CHART
# ============================================================

st.markdown("## 💨 Wind Speed Forecast")

wind_fig = go.Figure()

wind_fig.add_trace(go.Scatter(
    x=df["time"],
    y=df["wind_speed"],
    mode="lines",
    fill="tozeroy",
    name="Wind Speed"
))

wind_fig.update_layout(
    template="plotly_dark",
    height=500,
    paper_bgcolor="#111827",
    plot_bgcolor="#111827",
    margin=dict(l=20, r=20, t=40, b=20)
)

st.plotly_chart(wind_fig, use_container_width=True)

# ============================================================
# RAW DATA
# ============================================================

st.markdown("## 📄 Weather Dataset")

st.dataframe(df.tail(20))

# ============================================================
# DOWNLOAD REPORT
# ============================================================

csv = df.to_csv(index=False)

st.download_button(
    label="⬇️ Download Weather Report",
    data=csv,
    file_name="weather_report.csv",
    mime="text/csv"
)

"""

# ============================================================
# SAVE FILE
# ============================================================

with open("app.py", "w") as f:
    f.write(app_code)

# ============================================================
# START STREAMLIT
# ============================================================

from pyngrok import ngrok

# ------------------------------------------------------------
# PASTE YOUR REAL NGROK TOKEN
# ------------------------------------------------------------

ngrok.set_auth_token("3CRp2aAUrPM19FOZOIMHTKvUkEO_6QSpNDuRQG793V1N12h2V")

# ============================================================
# KILL OLD TUNNELS
# ============================================================

ngrok.kill()

# ============================================================
# START STREAMLIT SERVER
# ============================================================

get_ipython().system_raw(
    "streamlit run app.py --server.port 8501 &"
)

# ============================================================
# CREATE NGROK URL
# ============================================================

public_url = ngrok.connect(8501)

print("🚀 ADVANCED WEATHER DASHBOARD RUNNING")
print("🌍 OPEN THIS URL:")
print(public_url)

🚀 ADVANCED WEATHER DASHBOARD RUNNING
🌍 OPEN THIS URL:
NgrokTunnel: "https://user-saline-bagful.ngrok-free.dev" -> "http://localhost:8501"
